<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/chunking_strategies_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q huggingface_hub langchain_core langchain_huggingface torch tiktoken datasets "chonkie[semantic]" chonkie chromadb "bitsandbytes>=0.46.1"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from huggingface_hub import login


hf_token = ""

# Colab Secrets (🔑 in the sidebar)
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    pass


login(token=hf_token)

In [ ]:
from huggingface_hub import whoami
print(whoami())

In [ ]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.3"

dataset_dict = load_dataset(repo_id, token=hf_token)
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

Define chunkers

In [ ]:
from chonkie import (
    SemanticChunker,
    RecursiveChunker,
    RecursiveRules,
    RecursiveLevel,
    OverlapRefinery,
)

import re

# ---------- Chunkers ----------
rules = RecursiveRules(levels=[
    RecursiveLevel(delimiters=["\n\n"], include_delim="prev"),
    RecursiveLevel(delimiters=["\n"], include_delim="prev"),
    RecursiveLevel(delimiters=[". ", "! ", "? "], include_delim="prev"),
    RecursiveLevel(whitespace=True),
])

semantic_chunker = SemanticChunker(
    threshold=0.7,
    chunk_size=400,              # tokens
    similarity_window=3,
    min_sentences_per_chunk=2,
    skip_window=0,
)

recursive_chunker = RecursiveChunker(
    tokenizer="character",
    chunk_size=700,              # characters
    rules=rules,
)

overlap_refinery = OverlapRefinery(
    tokenizer="character",
    context_size=120,            # fixed overlap in characters
    mode="recursive",
    rules=rules,
    method="prefix",
    merge=True,
)


Define chunk functions

In [ ]:
def clean_text(text):
    # Stop "et al." from being treated as a sentence ending
    text = text.replace("et al.", "et al")
    # Remove image tags
    text = re.sub(r"\[?(?:<[^>]+\.webp>)+(?:-\d+)?\]?", "", text)
    return text

# ---------- Chunk functions----------
def perform_semantic_chunking(text):
    chunks = semantic_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]


def perform_structured_chunking(text):
    chunks = recursive_chunker.chunk(clean_text(text))
    return [c.text for c in overlap_refinery(chunks)]

Define vector database

In [ ]:
import chromadb
client = chromadb.PersistentClient(path="/content/drive/MyDrive/colab_projects/VGU_WS26_Project/chroma_db")

In [ ]:
def build_collection(records, name, rebuild=False):
    # records: list of {"document": str, "metadata": dict}
    if rebuild:
        try:
            client.delete_collection(name)
        except Exception:
            pass

    collection = client.get_or_create_collection(name)

    batch = client.get_max_batch_size()
    for start in range(0, len(records), batch):
        part = records[start:start + batch]
        collection.upsert(
            ids=[f"chunk_{start + i}" for i in range(len(part))],
            documents=[r["document"] for r in part],
            metadatas=[r["metadata"] for r in part],
        )

    print(f"{name}: built {collection.count()} chunks")
    return collection


def collection_ready(name):
  try:
      collection = client.get_collection(name)
  except Exception:
      return False
  return collection.count() > 0

Build records for ChromaDB

In [ ]:
def make_records(df, chunk_fn):
    records = []
    for row in df.itertuples():
        chunks = chunk_fn(row.case_text)
        for chunk_id, chunk in enumerate(chunks):
            records.append({
                "document": chunk,
                "metadata": {
                    "case_id": row.case_id,
                    "disease": row.disease,
                    "chunk_id": chunk_id,
                    "total_chunks": len(chunks),
                },
            })
    return records

semantic_records = make_records(train_df, perform_semantic_chunking)
structured_records = make_records(train_df, perform_structured_chunking)

In [ ]:
COLLECTION_NAME = "semantic_chunking"


if collection_ready(COLLECTION_NAME):
    collection = client.get_collection(COLLECTION_NAME)
    print("Reusing existing collection")
else:
    records = make_records(train_df, perform_semantic_chunking)
    collections = build_collection(records, "cases_semantic", rebuild=True)

RAG

In [ ]:
def retrieve(collection, query, k=3, search_k=20):
    # Search at chunk level
    res = collection.query(query_texts=[query], n_results=search_k)

    # Keep the best (closest) chunk per case
    best_by_case = {}
    for meta, dist in zip(res["metadatas"][0], res["distances"][0]):
        case_id = meta["case_id"]
        if case_id not in best_by_case or dist < best_by_case[case_id]["distance"]:
            best_by_case[case_id] = {
                "case_id": case_id,
                "disease": meta["disease"],
                "distance": dist,
            }

    # Rank cases, closest first, and keep the top k
    return sorted(best_by_case.values(), key=lambda r: r["distance"])[:k]


def attach_case_text(hits, df):
    # Look up the full case text for each retrieved case
    text_by_id = df.set_index("case_id")["case_text"]
    return [{**hit, "text": text_by_id.loc[hit["case_id"]]} for hit in hits]

In [ ]:
def attach_case_text(hits, df):
    # Look up the full case text for each retrieved case
    text_by_id = df.set_index("case_id")["case_text"]
    return [{**hit, "text": text_by_id.loc[hit["case_id"]]} for hit in hits]

In [ ]:
query = (
    "A 28-year-old man returning from a trip to rural Southeast Asia presents with high fever, "
    "severe joint pain, headache, and a rash that appeared two days after the fever started."
)

hits = retrieve(collection, query, k=3)
results = attach_case_text(hits, train_df)

for rank, result in enumerate(results, start=1):
    print(f"#{rank} | disease: {result['disease']} | case: {result['case_id']} | distance: {result['distance']:.3f}")
    print(result["text"][:300], "...\n")

MedGemma

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM, BitsAndBytesConfig

model_id = "google/medgemma-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(model_id, token=hf_token)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token,
)

In [ ]:
SYSTEM_PROMPT = (
    "You are a careful clinical assistant. Use ONLY the retrieved case reports below as evidence.\n"
    "You MUST answer with exactly one disease name, copied verbatim from this list, and nothing else:\n"
    + "\n".join(f"- {name}" for name in DISEASE_LABELS)
)


def generate(system_prompt, user_prompt, max_new_tokens=32):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False,
    ).to(model.device)

    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True
    )


def diagnose(question, collection, k=3, max_new_tokens=32):
    hits = retrieve(collection, question, k=k)
    retrieved = attach_case_text(hits, train_df)
    context = "\n\n".join(
        f"Case ({result['disease']}): {result['text']}" for result in retrieved
    )
    user_prompt = f"Retrieved cases:\n{context}\n\nPatient presentation:\n{question}"
    response = generate(SYSTEM_PROMPT, user_prompt, max_new_tokens=max_new_tokens)
    return response, retrieved

In [ ]:
question = (
    "A 28-year-old man returning from a trip to rural Southeast Asia presents with high fever, "
    "severe joint pain, headache, and a rash that appeared two days after the fever started."
)

answer, sources = diagnose(question, collection)

print("Model diagnosis:", answer)
print("Retrieved from cases:", [result["disease"] for result in sources])